# ChatGLM3 QLoRA 4-bit 微調實戰（2026 版）

## 學習目標

1. 理解 4-bit 量化（QLoRA）的原理，以及 `BitsAndBytesConfig` 的正確用法。
2. 掌握 `nf4` vs `fp4`、double quantization、`bf16` compute dtype 的取捨。
3. 使用 `tokenizer.apply_chat_template()` 組訊息，確保訓練與推論 prompt 格式一致。
4. 以 `trl.SFTTrainer` 完成指令微調，了解它如何自動處理 response-only `-100` 遮罩。
5. 正確的 PEFT 初始化順序：`BitsAndBytesConfig` → `prepare_model_for_kbit_training()` → `LoraConfig` → `get_peft_model()`。

## 前置知識
- 已完成 `04-kbits-tuning/04-1bits_training/` 的 8-bit 量化入門。
- 了解 LoRA 的基本概念（低秩分解、`r`、`lora_alpha`、`target_modules`）。

## 銜接說明
- **上一個 notebook**：`../04-3lora_training/chatglm3_lora.ipynb` — LoRA 全精度版本
- **下一個 notebook**：`../../05-Multimodal/` — 多模態訓練（也使用 `apply_chat_template` 與 `processor`）

## 硬體需求
- 4-bit QLoRA 單 GPU：ChatGLM3-6B 約需 **8 GB VRAM**（vs 全精度 bf16 約 14 GB）。
- 若 VRAM 不足，本 notebook 亦示範使用 `Qwen/Qwen2.5-1.5B-Instruct`（~4 GB VRAM）作為輕量替代。

In [ ]:
# ── 版本鎖定（建議在獨立虛擬環境中執行）──
# pip install \
#   "transformers>=4.46" \
#   "datasets>=3.0" \
#   "trl>=0.12" \
#   "peft>=0.13" \
#   "accelerate>=1.0" \
#   "bitsandbytes>=0.44" \
#   "evaluate>=0.4" \
#   "safetensors>=0.4" \
#   "torch>=2.4"

import transformers, datasets, trl, peft, accelerate, bitsandbytes, evaluate, safetensors, torch

print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"trl          : {trl.__version__}")
print(f"peft         : {peft.__version__}")
print(f"accelerate   : {accelerate.__version__}")
print(f"bitsandbytes : {bitsandbytes.__version__}")
print(f"evaluate     : {evaluate.__version__}")
print(f"safetensors  : {safetensors.__version__}")
print(f"torch        : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Step 1 — 匯入套件

2026 版核心套件說明：
- `BitsAndBytesConfig`：量化設定的統一載體，所有量化參數集中於此物件傳入 `from_pretrained`。
- `prepare_model_for_kbit_training`：量化模型在套用 LoRA 前必要的穩定化步驟。
- `trl.SFTTrainer` / `SFTConfig`：自動處理 response-only 損失遮罩，無需手刻 `-100` 標籤。
- `transformers.set_seed`：確保訓練可重現。

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

set_seed(42)

## Step 2 — 載入資料集

以 HuggingFace Hub 的 `load_dataset()` 載入資料集，並以環境變數或 `pathlib` 管理本機快取路徑，確保跨環境可執行。若已有本機副本，可將 `DATASET_ID` 換成本機路徑字串。

In [ ]:
import os
from pathlib import Path

# --- 設定：改這裡即可，不要硬寫路徑在 code 裡 ---
MODEL_ID = "THUDM/chatglm3-6b"           # HF Hub model id；離線時可改成本機路徑
DATASET_ID = "silk-road/alpaca-data-gpt4-chinese"  # HF Hub 中文 alpaca 資料集
OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", "./chatbot-qlora"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 輕量替代（VRAM < 8 GB 時取消下方注解）
# MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # ~4 GB VRAM，支援 apply_chat_template

# 載入資料集
ds = load_dataset(DATASET_ID, split="train")
print(ds)
print(ds[0])

## Step 3 — 資料集預處理

### 3.1 載入 Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,   # ChatGLM3 需要；Qwen2.5 不需要
    use_fast=True,
)
print(f"EOS token: {tokenizer.eos_token!r}  (id={tokenizer.eos_token_id})")
print(f"Chat template exists: {tokenizer.chat_template is not None}")

### 3.2 用 `apply_chat_template` 組訊息

`apply_chat_template` 是 transformers 4.35+ 的標準介面，每個 Hub 上的對話模型都有隨模型附帶的 `chat_template`（存在 `tokenizer_config.json` 內）。訓練時用同一模板組 prompt，推論時也用同一模板，確保訓練與推論的 prompt 格式分佈一致。

```python
# 2026 標準寫法（跨模型可攜）
messages = [{"role": "user", "content": query}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
```

> **多模態橋樑**：`apply_chat_template` 支援 `{"type": "image", "image": ...}` 等多模態內容項目，下一章 05-Multimodal 將直接沿用這套機制。

In [ ]:
def format_messages(example: dict) -> dict:
    """Convert alpaca-format record to a chat messages dict.

    Alpaca schema: instruction, input (optional), output.
    Returns a dict with key 'text' containing the full formatted string.
    SFTTrainer will use this 'text' field (or formatting_func) to build tokens.
    """
    # Build user message: combine instruction and (optional) input
    user_content = example["instruction"]
    if example.get("input", "").strip():
        user_content = f"{example['instruction']}\n{example['input']}"

    messages = [
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": example["output"]},
    ]
    # tokenize=False: return raw string; SFTTrainer handles tokenization
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,  # False during training: we provide the full response
    )
    return {"text": text}

# Map over the dataset (batched=True is ~3-5x faster via Arrow vectorisation)
formatted_ds = ds.map(format_messages, batched=False, num_proc=1, remove_columns=ds.column_names)
print(formatted_ds)
print(formatted_ds[0]["text"][:500])

### 3.3 手刻版前處理（底層原理：-100 遮罩）

這一段呈現底層手刻版，幫助理解 SFTTrainer 在背後做了什麼。**實際訓練用 SFTTrainer（Step 6），不需手刻此邏輯**。

**為什麼需要 -100 遮罩？**

語言模型訓練的損失函數（CrossEntropyLoss）對每個 token 位置都計算 loss。指令微調時，我們只希望模型學習「如何回應」，而不是學習「如何複述 prompt」。將 prompt 部分的標籤設成 `-100` 後，`CrossEntropyLoss` 會自動忽略這些位置（PyTorch 的 `ignore_index=-100` 預設）。

```
input_ids : [gMASK][sop][USER]...query...[ASST]  response  [EOS]
labels    :  -100   -100  -100 ... -100    -100   response  [EOS]
                    ← 只計算這段的 loss →
```

In [ ]:
def process_func_manual(example: dict, max_length: int = 256) -> dict:
    """Manual tokenization with -100 label masking for response-only training.

    Kept as a reference to show what SFTTrainer does internally.
    Do NOT use this together with SFTTrainer.
    """
    user_content = example["instruction"]
    if example.get("input", "").strip():
        user_content = f"{example['instruction']}\n{example['input']}"

    # Prompt portion (user turn only, with generation prompt)
    prompt_messages = [{"role": "user", "content": user_content}]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    prompt_tokens = tokenizer(prompt_text, add_special_tokens=False)

    # Response portion
    response_text = example["output"]
    response_tokens = tokenizer(response_text, add_special_tokens=False)

    # Concatenate: [prompt tokens] + [response tokens] + [EOS]
    input_ids = (
        prompt_tokens["input_ids"]
        + response_tokens["input_ids"]
        + [tokenizer.eos_token_id]
    )
    attention_mask = (
        prompt_tokens["attention_mask"]
        + response_tokens["attention_mask"]
        + [1]
    )
    # -100 for prompt positions; only response + EOS contribute to loss
    labels = (
        [-100] * len(prompt_tokens["input_ids"])
        + response_tokens["input_ids"]
        + [tokenizer.eos_token_id]
    )

    # Truncate to max_length
    return {
        "input_ids":      input_ids[:max_length],
        "attention_mask": attention_mask[:max_length],
        "labels":         labels[:max_length],
    }

# Sanity check on one example
example_out = process_func_manual(ds[1])
print("Prompt tokens (masked):", tokenizer.decode([i for i in example_out["input_ids"] if example_out["labels"][example_out["input_ids"].index(i)] == -100][:10]))
print("Response tokens:",        tokenizer.decode([i for i, l in zip(example_out["input_ids"], example_out["labels"]) if l != -100]))

## Step 4 — 建立量化模型

### 4.1 量化精度選擇指南

| 精度 | VRAM（6B 模型） | 訓練支援 | 推薦場景 |
|---|---|---|---|
| bf16（全精度） | ~14 GB | 是 | GPU >= A100/H100 |
| fp16（全精度） | ~14 GB | 是（但梯度易溢位） | 舊 GPU（不支援 bf16） |
| 8-bit（LLM.int8） | ~9 GB | 需 PEFT | 記憶體略緊 |
| **4-bit NF4（QLoRA）** | **~8 GB** | **需 PEFT** | **VRAM 緊張首選** |

### 4.2 BitsAndBytesConfig 詳解

量化設定統一透過 `BitsAndBytesConfig` 物件傳入 `from_pretrained`：

```python
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # bf16 而非 fp16
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
)
```

**關鍵概念說明**：

- **NF4 vs FP4**：NF4（Normal Float 4）是針對正態分佈權重最佳化的非均勻量化格式，對 LLM 權重的統計分佈匹配度更好，精度損失比 FP4 顯著更低。
- **Double Quantization**：對量化常數（scale factor）再做一次量化，每個參數額外節省約 0.37 bits，整體額外省 ~0.4 GB（6B 模型），計算 overhead 可忽略。
- **bf16 compute dtype**：bf16 與 fp32 動態範圍相同（±3.4×10³⁸），指數位元多 3 bits，在 A100/H100/RTX 30xx+ GPU 上計算速度一樣快，且不需要 loss scaling，是 4-bit 訓練的首選計算精度。
- **`device_map='auto'`**：Accelerate 自動依 VRAM 分配 layer 到 GPU/CPU/disk，已隱含 `low_cpu_mem_usage=True`（分片載入，不一次佔滿 CPU RAM）。
- **safetensors**：以 mmap 方式載入，不執行 pickle `__reduce__` 程式碼，無 arbitrary code execution 風險，且載入速度比 `.bin` 快 2-3 倍。

In [ ]:
# 4-bit NF4 量化設定
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # nf4 精度優於 fp4
    bnb_4bit_compute_dtype=torch.bfloat16,  # 計算時反量化至 bf16
    bnb_4bit_use_double_quant=True,      # 對量化常數再量化，多省 ~0.4 GB
)

# 載入模型（~8 GB VRAM for 6B; 輕量替代約 4 GB）
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",                  # 自動分配層到 GPU/CPU，隱含 low_cpu_mem_usage
    torch_dtype=torch.bfloat16,         # 非量化層（embedding、lm_head）使用 bf16
    use_safetensors=True,               # 安全且快速的權重格式
    trust_remote_code=True,             # ChatGLM3 需要；Qwen2.5 可移除
)
model.config.use_cache = False  # gradient checkpointing 與 KV cache 不相容
print(model)

In [ ]:
# 驗證各層的 dtype 分佈
for name, param in model.named_parameters():
    print(f"{name:<60s} {str(param.dtype):<20s} {str(param.device)}")

## Step 5 — LoRA 設定

### 5.1 正確的初始化順序

在量化模型上套用 LoRA 時，必須先執行 `prepare_model_for_kbit_training()`，再呼叫 `get_peft_model()`。

`prepare_model_for_kbit_training()` 做了以下三件事：
1. 將所有量化層的 `requires_grad` 設為 `False`（只有 LoRA adapter 訓練）。
2. 將 `LayerNorm` 升精度至 `fp32`，避免梯度流穿越量化 bottleneck 時精度過低。
3. 啟用 `input_require_grads`。

**正確順序**：
```
BitsAndBytesConfig → from_pretrained → prepare_model_for_kbit_training → LoraConfig → get_peft_model
```

In [ ]:
# Step 5.1: 穩定化量化模型（PEFT 前必要步驟）
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,  # 用 recompute activation 換 VRAM（約減少 40%）
)

# Step 5.2: LoRA 設定
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["query_key_value"],  # ChatGLM3 的注意力投影層
    # Qwen2.5 替代：target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
    r=8,                   # 低秩維度；r=4 更省，r=16 效果更好
    lora_alpha=32,         # scaling = lora_alpha / r；通常設 2*r
    lora_dropout=0.1,
    bias="none",           # 不訓練 bias（節省參數）
    inference_mode=False,
)

# Step 5.3: 套用 LoRA adapter
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 預期輸出：trainable params 約 1.8M / total 6B，約 0.03%

## Step 6 — 訓練

### 6.1 SFTTrainer 現代化路徑

`trl.SFTTrainer` 繼承自 `Trainer`，額外處理：
- 自動以 `formatting_func` 或 `text` 欄位組成訓練文本。
- 自動呼叫 `DataCollatorForCompletionOnlyLM`，僅對 assistant 回應計算 loss，無需手刻 `-100` 遮罩。
- 支援 sequence packing（多筆短序列拼一個 context window），大幅提升 GPU 利用率。

**effective batch size = per_device_batch_size × gradient_accumulation_steps**：
本設定為 `1 × 32 = 32`，在單 GPU 上模擬 batch size 32 的效果，同時保持低 peak VRAM。

In [ ]:
sft_config = SFTConfig(
    # --- 輸出 ---
    output_dir=str(OUTPUT_DIR),

    # --- 訓練規模 ---
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,   # effective batch = 1 * 32 = 32
    gradient_checkpointing=True,      # 以重新計算換 VRAM
    max_seq_length=256,

    # --- 優化器 ---
    learning_rate=1e-4,
    optim="paged_adamw_32bit",        # bitsandbytes 的 paged optimizer，省 VRAM
    # optim="adamw_torch_fused",      # 若不用 bnb optimizer 可換這個
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,

    # --- 精度 ---
    bf16=True,                        # bf16 訓練（需 Ampere+ GPU）
    # fp16=True,                      # 舊 GPU 改這行

    # --- 日誌 / 儲存 ---
    logging_steps=10,
    save_strategy="epoch",
    save_safetensors=True,            # 以 safetensors 格式儲存 adapter

    # --- 資料欄位 ---
    dataset_text_field="text",        # 對應 formatted_ds 的 'text' 欄位

    # --- 可重現性 ---
    seed=42,
    data_seed=42,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=formatted_ds.select(range(6000)),  # 取前 6000 筆與原版一致
    processing_class=tokenizer,
    peft_config=lora_config,          # SFTTrainer 自動套用 LoRA adapter
)

print("Trainer ready. Starting training...")

## Step 7 — 執行訓練與儲存

In [ ]:
trainer.train()

# 儲存 LoRA adapter（safe_serialization=True 預設已由 save_safetensors=True 控制）
trainer.save_model(str(OUTPUT_DIR))
print(f"LoRA adapter saved to {OUTPUT_DIR}")

### 7.1 （選用）推送至 HuggingFace Hub

模型訓練完後，可以直接推送至 HF Hub 與社群分享，並附上最小 model card。

In [ ]:
# 取消下方注解以推送至 HF Hub
# import os
# HUB_MODEL_ID = "your-username/chatglm3-6b-qlora-alpaca-zh"
# trainer.push_to_hub(
#     hub_model_id=HUB_MODEL_ID,
#     commit_message="Add QLoRA 4-bit adapter trained on alpaca-data-zh",
#     tags=["chatglm3", "qlora", "4-bit", "alpaca", "zh", "instruction-tuning"],
#     language=["zh"],
#     license="apache-2.0",
# )
# print(f"Model pushed to https://huggingface.co/{HUB_MODEL_ID}")
print("push_to_hub is commented out. Uncomment and set HUB_MODEL_ID to upload.")

## Step 8 — 模型推理

### 8.1 推論時的 apply_chat_template

推論時使用與訓練完全相同的 `apply_chat_template`，確保 prompt 格式一致。`add_generation_prompt=True` 會在最後加上 assistant turn 的起始 token，引導模型生成回應。

In [ ]:
def chat_inference(model, tokenizer, user_message: str, max_new_tokens: int = 256) -> str:
    """Run inference using the standard apply_chat_template interface.

    Works with any model that has a chat_template in its tokenizer config.
    """
    model.eval()
    messages = [{"role": "user", "content": user_message}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # True during inference: prompt model to generate response
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,  # avoid warning when pad_token not set
        )
    # Decode only the newly generated tokens (skip the prompt)
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


response = chat_inference(model, tokenizer, "數學考試怎麼考高分？")
print(response)

### 8.2 載入已儲存的 Adapter（推薦部署方式）

訓練完成後，adapter 與 base model 分開儲存。部署時可獨立載入 adapter，保持 base model 不變。

In [ ]:
# 重新載入 base model + 合併 adapter（示範部署流程）
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
    trust_remote_code=True,
)

# 套用已訓練的 LoRA adapter
loaded_model = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR))
loaded_model.eval()

print("Adapter loaded. Running inference...")
response = chat_inference(loaded_model, tokenizer, "怎麼學好英文？")
print(response)

## 小結

本 notebook 完整示範了 2026 版 QLoRA 指令微調流程：

- **量化設定**：以 `BitsAndBytesConfig` 集中管理所有量化參數，透過 `quantization_config=` 傳入 `from_pretrained`。
- **計算精度**：`bnb_4bit_compute_dtype=torch.bfloat16`，動態範圍寬、無需 loss scaling。
- **模型載入**：`device_map='auto'` 自動分配層到可用裝置，搭配 `use_safetensors=True` 安全快速載入。
- **PEFT 前置**：`prepare_model_for_kbit_training()` 必須在 `get_peft_model()` 之前執行，穩定量化層梯度流。
- **對話格式**：`apply_chat_template()` 統一訓練與推論的 prompt 格式，具備跨模型可攜性。
- **訓練框架**：`SFTTrainer` + `SFTConfig` 自動處理 response-only 損失遮罩，無需手刻 `-100` 標籤。
- **儲存格式**：`save_safetensors=True` 以 safetensors 格式儲存 adapter，安全且載入快速。
- **可重現性**：`set_seed(42)` 確保每次訓練結果一致。

## 練習題

1. 將 `bnb_4bit_quant_type` 從 `nf4` 改為 `fp4`，重新訓練後比較 loss 曲線，觀察精度差異。
2. 關閉 `bnb_4bit_use_double_quant=True`，用 `nvidia-smi` 量測 VRAM 差異（理論約 0.4 GB）。
3. 將 `r=8` 改為 `r=16` 或 `r=32`，觀察 `trainable parameters` 數量的變化及訓練速度。
4. 以 `Qwen/Qwen2.5-1.5B-Instruct` 取代 ChatGLM3，調整 `target_modules`（提示：用 `model.named_modules()` 找注意力層名稱），驗證 `apply_chat_template` 的跨模型可攜性。
5. 在 `chat_inference` 中加入 `repetition_penalty=1.2`，觀察生成品質的變化。